# Problem 04: 
Sử dụng thư viện Machine Learning (Sklearn hoặc Skorch) thực thi lại phương pháp Linear Regression.

In [85]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [86]:
df = pd.read_csv('forestfires.csv')

In [87]:
#  Covert days into numerical values

def convert_day(day: str)->int:
    day_dict = {
        'mon' : 1,
        'tue' : 2,
        'wed' : 3,
        'thu' : 4,
        'fri' : 5,
        'sat' : 6,
        'sun' : 7,
    }
    return day_dict[day]
# Covert months into numerical values

def convert_month(month: str)->int:
    month_dict = {
        "jan" : 1,
        "feb" : 2,
        "mar" : 3,
        "apr" : 4,
        "may" : 5,
        "jun" : 6,
        "jul" : 7,
        "aug" : 8,
        "sep" : 9,
        "oct" : 10,
        "nov" : 11,
        "dec" : 12,
    }
    return month_dict[month]
df['day'] = df['day'].apply(convert_day)
df['month'] = df['month'].apply(convert_month)
df.head(5)

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,3,5,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,10,2,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,10,6,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,3,5,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,3,7,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [88]:
# Pick out features
X = df.drop(columns = ['area'])
Y = df['area']
# Divide dataset into training data and testing data (80/20)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [89]:
# Drop feature with high correlation rate
corr_matrix = X_train.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.8)]

print("Features dropped due to high correlation:", to_drop)

# Make sure testing and training data are at the same size
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

Features dropped due to high correlation: ['DC']


In [90]:
# Standardize using sklearn
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
columns_to_standardize = ['FFMC','DMC','ISI', 'temp', 'RH', 'wind', 'rain']
df[columns_to_standardize] = scaler.fit_transform(df[columns_to_standardize])

In [91]:
# Using log transformation to normalize because 'area' (skewed - lean toward 0.0) is not following normal distribution
Y_train = np.log1p(Y_train)
Y_test = np.log1p(Y_test)

In [92]:
class MyLinearRegression:
    def __init__(self):
        # Automatically add bias value
        self.model = LinearRegression(fit_intercept=True)
        
    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self.model.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)

    def rmse(self, y: np.ndarray, y_hat: np.ndarray) -> float:
        return np.sqrt(mean_squared_error(y, y_hat))

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> float:
        y_pred = self.predict(X)
        return self.rmse(y, y_pred)

In [93]:
model = MyLinearRegression()
# Training model
model.fit(X_train, Y_train)

# Predict in testing dataset
y_pred = model.predict(X_test)

# Evaluate model
rmse_value = model.evaluate(X_test, Y_test)
print(f'RMSE: {rmse_value:.6f}')


RMSE: 1.473351


In [95]:
a = 1.546710 - 1.473351
print(f'{a:.6f}')

0.073359


# Nhận xét:
- Có sự khác nhau nhỏ tại giá trị RMSE khi sử dụng numpy và khi sử dung Sklearn (1.546710 - 1.473351 = 0.073359)
- Giải thích: 
  + Numpy chia dataset theo thứ tự (split dựa trên số dòng của file)
  + Sklearn sử dụng phân chia ngẫu nhiên (random_state = 42)
  + Numpy tính toán trực tiếp trên công thức không thêm bias (1) vào cột X, ngược lại với khi sử dụng Sklearn